In [1]:
# ==============================================================================
# CELL: Simple Buffer Visualization Engine (Zero-Cost Version)
# ==============================================================================
# Objective: For a given coordinate, generate a 500-meter buffer and display
# it on an interactive map. This script makes ZERO API calls.

# --- 0. INSTALL/UPGRADE DEPENDENCIES ---
!pip install geopandas folium mapclassify -q

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

print("--- Simple Buffer Visualization Engine (Zero-Cost) Initializing ---")

# --- 1. CONFIGURATION (ARCHITECT'S INPUT) ---
# === ARCHITECT'S ACTION REQUIRED ===
# Enter the precise latitude and longitude for the center point.
CENTER_LAT = 19.3622296  # Example for "Carlos Lazo 425, Santa Fe"
CENTER_LON = -99.2661372 # Example for "Carlos Lazo 425, Santa Fe"
# ===================================

# The radius of the circle in meters.
BUFFER_RADIUS_METERS = 500

# --- 2. CREATE THE BUFFER GEOMETRY (OFFLINE OPERATION) ---
print("\nStep 1: Generating 500-meter buffer from provided coordinates...")
try:
    # Create a GeoDataFrame with our single point. CRS is WGS84 (lat/lon).
    point_gdf = gpd.GeoDataFrame(
        [{'geometry': Point(CENTER_LON, CENTER_LAT)}],
        crs='EPSG:4326'
    )

    # Re-project to a meter-based CRS (UTM Zone 14N for CDMX).
    point_gdf_utm = point_gdf.to_crs(epsg=32614)

    # Create the buffer using meters.
    buffer_gdf_utm = point_gdf_utm.buffer(BUFFER_RADIUS_METERS)

    # Convert the buffer back to lat/lon for map display.
    buffer_gdf = buffer_gdf_utm.to_crs(epsg=4326)

    final_buffer_gdf = gpd.GeoDataFrame(geometry=buffer_gdf, crs='EPSG:4326')
    print(" -> Buffer generation complete.")

    # --- 3. VISUALIZE THE RESULT ---
    print("\nStep 2: Generating interactive map...")
    # Create the base map centered on the point
    m = final_buffer_gdf.explore(
        location=[CENTER_LAT, CENTER_LON],
        zoom_start=15,
        color="blue",
        style_kwds={"fillOpacity": 0.3}
    )

    # Add the center point on top
    point_gdf.explore(
        m=m,
        color="red",
        marker_kwds={"radius": 8, "tooltip": f"Center:\n({CENTER_LAT}, {CENTER_LON})"}
    )

    display(m)
    print("\n--- Visualization Complete ---")
    print("The map above shows the specified center point and the 500-meter radius around it.")

except Exception as e:
    print(f"An error occurred during the geometric operation: {e}")

--- Simple Buffer Visualization Engine (Zero-Cost) Initializing ---

Step 1: Generating 500-meter buffer from provided coordinates...
 -> Buffer generation complete.

Step 2: Generating interactive map...



--- Visualization Complete ---
The map above shows the specified center point and the 500-meter radius around it.


In [3]:
# ==============================================================================
# CELL: The Definitive Jittering Engine v2.2 (Hardened & Patched)
# ==============================================================================
# Objective: Jitter points on a specific street found ONLY within a local
# search area, with hardened filtering logic to prevent NaN errors.

# --- 0. INSTALL/UPGRADE DEPENDENCIES ---
!pip install osmnx geopandas folium mapclassify -q

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import osmnx as ox
import os
import random
from google.colab import drive

print("--- Local Search Area Jittering Engine (v2.2) Initializing ---")

# --- 1. CONFIGURATION ---
drive.mount('/content/drive')

# === ARCHITECT'S ACTION REQUIRED ===
# CORRECTED COORDINATES HAVE BEEN IMPLEMENTED
CENTER_LAT = 19.3622296
CENTER_LON = -99.2661372
BUFFER_RADIUS_METERS = 1000
NUMBER_OF_POINTS_TO_GENERATE = 20
TARGET_STREET_NAME = "Avenida Santa Fe"
# ===================================

# --- CANONICAL PATHING DOCTRINE ---
CANONICAL_PATH = '/content/drive/My Drive/_Pienza/Assets/Phase_1'
HTML_OUTPUT_FILE = os.path.join(CANONICAL_PATH, 'jittering_verification_map_v2.html')
os.makedirs(CANONICAL_PATH, exist_ok=True)

# --- 2. ACQUIRE THE LOCAL ROAD NETWORK ---
print(f"\nStep 1: Acquiring local road network within a {BUFFER_RADIUS_METERS}m radius...")
try:
    G = ox.graph_from_point((CENTER_LAT, CENTER_LON), dist=BUFFER_RADIUS_METERS, network_type='drive')
    local_streets_gdf = ox.graph_to_gdfs(G, nodes=False, edges=True)
    print(" -> Success. Local road network downloaded.")
except Exception as e:
    raise RuntimeError(f"CRITICAL FAILURE: Could not fetch local road network. Details: {e}")

# --- 3. TARGETED STREET SEARCH (HARDENED) ---
print(f"\nStep 2: Searching for '{TARGET_STREET_NAME}' within the local network...")
if 'name' not in local_streets_gdf.columns:
    raise KeyError("'name' column not found in the downloaded street data.")

# CRITICAL FIX: The .str.contains() method is now hardened with `na=False`.
# This instructs pandas to treat any unnamed road (NaN) as a non-match,
# preventing the "Cannot mask with non-boolean array" error.
target_street_gdf = local_streets_gdf[
    local_streets_gdf['name'].str.contains(TARGET_STREET_NAME, case=False, na=False)
]

if target_street_gdf.empty:
    raise ValueError(f"CRITICAL FAILURE: No segments of '{TARGET_STREET_NAME}' were found within the {BUFFER_RADIUS_METERS}m buffer. Halting.")

target_street_geom = target_street_gdf.geometry.unary_union
print(" -> Success. Target street segments have been isolated.")

# --- 4. GENERATE JITTERED POINTS ON THE TARGET SEGMENTS ---
print(f"\nStep 3: Generating {NUMBER_OF_POINTS_TO_GENERATE} random points...")
def get_random_point_on_line(line_geometry):
    if line_geometry.geom_type == 'MultiLineString':
        line_lengths = [line.length for line in line_geometry.geoms]
        chosen_line = random.choices(line_geometry.geoms, weights=line_lengths, k=1)[0]
    else:
        chosen_line = line_geometry
    random_distance = random.uniform(0, chosen_line.length)
    return chosen_line.interpolate(random_distance)

random_points = [get_random_point_on_line(target_street_geom) for _ in range(NUMBER_OF_POINTS_TO_GENERATE)]
print(" -> Point generation complete.")

# --- 5. VISUALIZE FOR VERIFICATION & EXPORT ---
print("\nStep 4: Generating interactive map and exporting to HTML...")
center_point_gdf = gpd.GeoDataFrame([{'geometry': Point(CENTER_LON, CENTER_LAT)}], crs='EPSG:4326')
target_street_display_gdf = gpd.GeoDataFrame(geometry=[target_street_geom], crs='EPSG:4326')
points_gdf_display = gpd.GeoDataFrame(geometry=random_points, crs='EPSG:4326')

m = local_streets_gdf.explore(color="gray", style_kwds={"weight": 1}, name="Local Network")
target_street_display_gdf.explore(m=m, color="blue", style_kwds={"weight": 5}, name="Target Street")
points_gdf_display.explore(m=m, color="red", marker_kwds={"radius": 5}, name="Jittered Points")
center_point_gdf.explore(m=m, color="yellow", marker_kwds={"radius": 8}, name="Center Point")

try:
    m.save(HTML_OUTPUT_FILE)
    print(f" -> SUCCESS: Interactive map has been saved as a standalone HTML file.")
    print(f" -> Location: {HTML_OUTPUT_FILE}")
except Exception as e:
    print(f" -> ERROR saving HTML file: {e}")

display(m)
print("\n--- Visualization Complete ---")
print("Map shows: Center (Yellow), Local Network (Gray), Target Street (Blue), Jittered Points (Red)")

--- Local Search Area Jittering Engine (v2.2) Initializing ---
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Step 1: Acquiring local road network within a 1000m radius...
 -> Success. Local road network downloaded.

Step 2: Searching for 'Avenida Santa Fe' within the local network...
 -> Success. Target street segments have been isolated.

Step 3: Generating 20 random points...
 -> Point generation complete.

Step 4: Generating interactive map and exporting to HTML...


/tmp/ipython-input-3667415165.py:61: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  target_street_geom = target_street_gdf.geometry.unary_union


 -> SUCCESS: Interactive map has been saved as a standalone HTML file.
 -> Location: /content/drive/My Drive/_Pienza/Assets/Phase_1/jittering_verification_map_v2.html



--- Visualization Complete ---
Map shows: Center (Yellow), Local Network (Gray), Target Street (Blue), Jittered Points (Red)


In [7]:
# ==============================================================================
# CELL: Simple Buffer Visualization Engine (Zero-Cost Version)
# ==============================================================================
# Objective: For a given coordinate, generate a 500-meter buffer and display
# it on an interactive map. This script makes ZERO API calls.

# --- 0. INSTALL/UPGRADE DEPENDENCIES ---
!pip install geopandas folium mapclassify -q

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

print("--- Simple Buffer Visualization Engine (Zero-Cost) Initializing ---")

# --- 1. CONFIGURATION (ARCHITECT'S INPUT) ---
# === ARCHITECT'S ACTION REQUIRED ===
# Enter the precise latitude and longitude for the center point.
CENTER_LAT = 19.4115574  # Example for "Carlos Lazo 425, Santa Fe"
CENTER_LON = -99.1705219 # Example for "Carlos Lazo 425, Santa Fe"
# ===================================

# The radius of the circle in meters.
BUFFER_RADIUS_METERS = 500

# --- 2. CREATE THE BUFFER GEOMETRY (OFFLINE OPERATION) ---
print("\nStep 1: Generating 500-meter buffer from provided coordinates...")
try:
    # Create a GeoDataFrame with our single point. CRS is WGS84 (lat/lon).
    point_gdf = gpd.GeoDataFrame(
        [{'geometry': Point(CENTER_LON, CENTER_LAT)}],
        crs='EPSG:4326'
    )

    # Re-project to a meter-based CRS (UTM Zone 14N for CDMX).
    point_gdf_utm = point_gdf.to_crs(epsg=32614)

    # Create the buffer using meters.
    buffer_gdf_utm = point_gdf_utm.buffer(BUFFER_RADIUS_METERS)

    # Convert the buffer back to lat/lon for map display.
    buffer_gdf = buffer_gdf_utm.to_crs(epsg=4326)

    final_buffer_gdf = gpd.GeoDataFrame(geometry=buffer_gdf, crs='EPSG:4326')
    print(" -> Buffer generation complete.")

    # --- 3. VISUALIZE THE RESULT ---
    print("\nStep 2: Generating interactive map...")
    # Create the base map centered on the point
    m = final_buffer_gdf.explore(
        location=[CENTER_LAT, CENTER_LON],
        zoom_start=15,
        color="blue",
        style_kwds={"fillOpacity": 0.3}
    )

    # Add the center point on top
    point_gdf.explore(
        m=m,
        color="red",
        marker_kwds={"radius": 8, "tooltip": f"Center:\n({CENTER_LAT}, {CENTER_LON})"}
    )

    display(m)
    print("\n--- Visualization Complete ---")
    print("The map above shows the specified center point and the 500-meter radius around it.")

except Exception as e:
    print(f"An error occurred during the geometric operation: {e}")

--- Simple Buffer Visualization Engine (Zero-Cost) Initializing ---

Step 1: Generating 500-meter buffer from provided coordinates...
 -> Buffer generation complete.

Step 2: Generating interactive map...



--- Visualization Complete ---
The map above shows the specified center point and the 500-meter radius around it.


In [10]:
# ==============================================================================
# CELL: The Definitive Jittering Engine v2.2 (Hardened & Patched)
# ==============================================================================
# Objective: Jitter points on a specific street found ONLY within a local
# search area, with hardened filtering logic to prevent NaN errors.

# --- 0. INSTALL/UPGRADE DEPENDENCIES ---
!pip install osmnx geopandas folium mapclassify -q

import pandas as pd
import os
import geopandas as gpd
from shapely.geometry import Point
import osmnx as ox
import random
from google.colab import drive

print("--- Local Search Area Jittering Engine (v2.2) Initializing ---")

# --- 1. CONFIGURATION ---
drive.mount('/content/drive')

# === ARCHITECT'S ACTION REQUIRED ===
# CORRECTED COORDINATES HAVE BEEN IMPLEMENTED
CENTER_LAT = 19.4115574
CENTER_LON = -99.1705219
BUFFER_RADIUS_METERS = 1000
NUMBER_OF_POINTS_TO_GENERATE = 20
TARGET_STREET_NAME = "Avenida México"
# ===================================

# --- CANONICAL PATHING DOCTRINE ---
CANONICAL_PATH = '/content/drive/My Drive/_Pienza/Assets/Phase_1'
HTML_OUTPUT_FILE = os.path.join(CANONICAL_PATH, 'jittering_verification_map_v3.html')
os.makedirs(CANONICAL_PATH, exist_ok=True)

# --- 2. ACQUIRE THE LOCAL ROAD NETWORK ---
print(f"\nStep 1: Acquiring local road network within a {BUFFER_RADIUS_METERS}m radius...")
try:
    G = ox.graph_from_point((CENTER_LAT, CENTER_LON), dist=BUFFER_RADIUS_METERS, network_type='drive')
    local_streets_gdf = ox.graph_to_gdfs(G, nodes=False, edges=True)
    print(" -> Success. Local road network downloaded.")
except Exception as e:
    raise RuntimeError(f"CRITICAL FAILURE: Could not fetch local road network. Details: {e}")

# --- 3. TARGETED STREET SEARCH (HARDENED) ---
print(f"\nStep 2: Searching for '{TARGET_STREET_NAME}' within the local network...")
if 'name' not in local_streets_gdf.columns:
    raise KeyError("'name' column not found in the downloaded street data.")

# CRITICAL FIX: The .str.contains() method is now hardened with `na=False`.
# This instructs pandas to treat any unnamed road (NaN) as a non-match,
# preventing the "Cannot mask with non-boolean array" error.
target_street_gdf = local_streets_gdf[
    local_streets_gdf['name'].str.contains(TARGET_STREET_NAME, case=False, na=False)
]

if target_street_gdf.empty:
    raise ValueError(f"CRITICAL FAILURE: No segments of '{TARGET_STREET_NAME}' were found within the {BUFFER_RADIUS_METERS}m buffer. Halting.")

target_street_geom = target_street_gdf.geometry.unary_union
print(" -> Success. Target street segments have been isolated.")

# --- 4. GENERATE JITTERED POINTS ON THE TARGET SEGMENTS ---
print(f"\nStep 3: Generating {NUMBER_OF_POINTS_TO_GENERATE} random points...")
def get_random_point_on_line(line_geometry):
    if line_geometry.geom_type == 'MultiLineString':
        line_lengths = [line.length for line in line_geometry.geoms]
        chosen_line = random.choices(line_geometry.geoms, weights=line_lengths, k=1)[0]
    else:
        chosen_line = line_geometry
    random_distance = random.uniform(0, chosen_line.length)
    return chosen_line.interpolate(random_distance)

random_points = [get_random_point_on_line(target_street_geom) for _ in range(NUMBER_OF_POINTS_TO_GENERATE)]
print(" -> Point generation complete.")

# --- 5. VISUALIZE FOR VERIFICATION & EXPORT ---
print("\nStep 4: Generating interactive map and exporting to HTML...")
center_point_gdf = gpd.GeoDataFrame([{'geometry': Point(CENTER_LON, CENTER_LAT)}], crs='EPSG:4326')
target_street_display_gdf = gpd.GeoDataFrame(geometry=[target_street_geom], crs='EPSG:4326')
points_gdf_display = gpd.GeoDataFrame(geometry=random_points, crs='EPSG:4326')

m = local_streets_gdf.explore(color="gray", style_kwds={"weight": 1}, name="Local Network")
target_street_display_gdf.explore(m=m, color="blue", style_kwds={"weight": 5}, name="Target Street")
points_gdf_display.explore(m=m, color="red", marker_kwds={"radius": 5}, name="Jittered Points")
center_point_gdf.explore(m=m, color="yellow", marker_kwds={"radius": 8}, name="Center Point")

try:
    m.save(HTML_OUTPUT_FILE)
    print(f" -> SUCCESS: Interactive map has been saved as a standalone HTML file.")
    print(f" -> Location: {HTML_OUTPUT_FILE}")
except Exception as e:
    print(f" -> ERROR saving HTML file: {e}")

display(m)
print("\n--- Visualization Complete ---")
print("Map shows: Center (Yellow), Local Network (Gray), Target Street (Blue), Jittered Points (Red)")

--- Local Search Area Jittering Engine (v2.2) Initializing ---
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Step 1: Acquiring local road network within a 1000m radius...
 -> Success. Local road network downloaded.

Step 2: Searching for 'Avenida México' within the local network...
 -> Success. Target street segments have been isolated.

Step 3: Generating 20 random points...
 -> Point generation complete.

Step 4: Generating interactive map and exporting to HTML...


/tmp/ipython-input-2695383187.py:61: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  target_street_geom = target_street_gdf.geometry.unary_union


 -> SUCCESS: Interactive map has been saved as a standalone HTML file.
 -> Location: /content/drive/My Drive/_Pienza/Assets/Phase_1/jittering_verification_map_v3.html



--- Visualization Complete ---
Map shows: Center (Yellow), Local Network (Gray), Target Street (Blue), Jittered Points (Red)


In [20]:
# ==============================================================================
# CELL: The Definitive Jittering Engine v2.5 (Hardened against List Types)
# ==============================================================================
# Objective: A hardened engine that pre-processes OSM data to handle list-type
# names, preventing 'unhashable type' errors before fuzzy matching.

# --- 0. INSTALL/UPGRADE DEPENDENCIES ---
!pip install osmnx geopandas folium mapclassify thefuzz python-Levenshtein -q

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import osmnx as ox
import random
from thefuzz import process
from google.colab import drive
import os # <-- Import the 'os' module

print("--- Fuzzy Matching Jittering Engine (v2.5) Initializing ---")

# --- 1. CONFIGURATION ---
drive.mount('/content/drive')

# === ARCHITECT'S ACTION REQUIRED ===
CENTER_LAT = 19.4115574
CENTER_LON = -99.1705219
BUFFER_RADIUS_METERS = 1000
NUMBER_OF_POINTS_TO_GENERATE = 20
TARGET_STREET_NAME = "av mexico, condesa / roma"
SCORE_CUTOFF = 50
# ===================================

CANONICAL_PATH = '/content/drive/My Drive/_Pienza/Assets/Phase_1'
HTML_OUTPUT_FILE = os.path.join(CANONICAL_PATH, 'fuzzy_match_verification_map_v2.html')
os.makedirs(CANONICAL_PATH, exist_ok=True)

# --- 2. ACQUIRE THE LOCAL ROAD NETWORK ---
print(f"\nStep 1: Acquiring local road network within a {BUFFER_RADIUS_METERS}m radius...")
try:
    G = ox.graph_from_point((CENTER_LAT, CENTER_LON), dist=BUFFER_RADIUS_METERS, network_type='drive')
    local_streets_gdf = ox.graph_to_gdfs(G, nodes=False, edges=True)
    print(" -> Success. Local road network downloaded.")
except Exception as e:
    raise RuntimeError(f"CRITICAL FAILURE: Could not fetch local road network. Details: {e}")

# --- 3. FUZZY MATCHING LOGIC (HARDENED) ---
print(f"\nStep 2: Finding the best fuzzy match for '{TARGET_STREET_NAME}'...")
if 'name' not in local_streets_gdf.columns:
    raise KeyError("'name' column not found in the downloaded street data.")

# --- CRITICAL FIX: DATA HARDENING STEP ---
# This pre-processes the 'name' column to handle list-type entries.
# It checks each entry: if it's a list, it takes the first item; otherwise, it keeps the original string.
cleaned_names_series = local_streets_gdf['name'].apply(lambda x: x[0] if isinstance(x, list) else x)
# Now, create the "lineup" from this clean, guaranteed-string data.
available_names = cleaned_names_series.dropna().unique().tolist()
# -----------------------------------------

# Use thefuzz to find the single best match and its score
best_match, score = process.extractOne(TARGET_STREET_NAME, available_names)

if score < SCORE_CUTOFF:
    raise ValueError(
        f"CRITICAL FAILURE: No high-confidence match found for '{TARGET_STREET_NAME}'.\n"
        f"The best match was '{best_match}' with a score of only {score}%.\n"
        f"Please check your TARGET_STREET_NAME or lower the SCORE_CUTOFF."
    )

print(f" -> Best match found: '{best_match}' (Score: {score}%)")
# Use the *best match* to query the *cleaned* series for the precise query
target_street_gdf = local_streets_gdf[cleaned_names_series == best_match]
target_street_geom = target_street_gdf.geometry.unary_union

# --- 4. GENERATE JITTERED POINTS ---
print(f"\nStep 3: Generating points on the matched street geometry...")
def get_random_point_on_line(line_geometry):
    if line_geometry.geom_type == 'MultiLineString':
        line_lengths = [line.length for line in line_geometry.geoms]
        chosen_line = random.choices(line_geometry.geoms, weights=line_lengths, k=1)[0]
    else: chosen_line = line_geometry
    return chosen_line.interpolate(random.uniform(0, chosen_line.length))

random_points = [get_random_point_on_line(target_street_geom) for _ in range(NUMBER_OF_POINTS_TO_GENERATE)]
print(" -> Point generation complete.")

# --- 5. VISUALIZE FOR VERIFICATION & EXPORT ---
print("\nStep 4: Generating interactive map and exporting to HTML...")
center_point_gdf = gpd.GeoDataFrame([{'geometry': Point(CENTER_LON, CENTER_LAT)}], crs='EPSG:4326')
target_street_display_gdf = gpd.GeoDataFrame(geometry=[target_street_geom], crs='EPSG:4326')
points_gdf_display = gpd.GeoDataFrame(geometry=random_points, crs='EPSG:4326')
m = local_streets_gdf.explore(color="gray", style_kwds={"weight": 1}, name="Local Network")
target_street_display_gdf.explore(m=m, color="blue", style_kwds={"weight": 5}, name=f"Best Match: {best_match}")
points_gdf_display.explore(m=m, color="red", marker_kwds={"radius": 5}, name="Jittered Points")
center_point_gdf.explore(m=m, color="yellow", marker_kwds={"radius": 8}, name="Center Point")
try:
    m.save(HTML_OUTPUT_FILE)
    print(f" -> SUCCESS: Interactive map has been saved to: {HTML_OUTPUT_FILE}")
except Exception as e:
    print(f" -> ERROR saving HTML file: {e}")
display(m)
print("\n--- Visualization Complete ---")

--- Fuzzy Matching Jittering Engine (v2.5) Initializing ---
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Step 1: Acquiring local road network within a 1000m radius...
 -> Success. Local road network downloaded.

Step 2: Finding the best fuzzy match for 'av mexico, condesa / roma'...
 -> Best match found: 'Avenida México' (Score: 65%)

Step 3: Generating points on the matched street geometry...
 -> Point generation complete.

Step 4: Generating interactive map and exporting to HTML...


/tmp/ipython-input-3342417444.py:72: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  target_street_geom = target_street_gdf.geometry.unary_union


 -> SUCCESS: Interactive map has been saved to: /content/drive/My Drive/_Pienza/Assets/Phase_1/fuzzy_match_verification_map_v2.html



--- Visualization Complete ---


In [25]:
# ==============================================================================
# CELL: The Definitive Jittering Engine v2.5 (Hardened against List Types)
# ==============================================================================
# Objective: A hardened engine that pre-processes OSM data to handle list-type
# names, preventing 'unhashable type' errors before fuzzy matching.

# --- 0. INSTALL/UPGRADE DEPENDENCIES ---
!pip install osmnx geopandas folium mapclassify thefuzz python-Levenshtein -q

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import osmnx as ox
import random
from thefuzz import process
from google.colab import drive
import os # <-- Import the 'os' module

print("--- Fuzzy Matching Jittering Engine (v2.5) Initializing ---")

# --- 1. CONFIGURATION ---
drive.mount('/content/drive')

# === ARCHITECT'S ACTION REQUIRED ===
CENTER_LAT = 19.3878315
CENTER_LON = -99.2501002
BUFFER_RADIUS_METERS = 3000
NUMBER_OF_POINTS_TO_GENERATE = 20
TARGET_STREET_NAME = "Bosques de Alisos, Bosques, Ciudad de Mexico, CDMX, Mexico"
SCORE_CUTOFF = 50
# ===================================

CANONICAL_PATH = '/content/drive/My Drive/_Pienza/Assets/Phase_1'
HTML_OUTPUT_FILE = os.path.join(CANONICAL_PATH, 'fuzzy_match_verification_map_v2.html')
os.makedirs(CANONICAL_PATH, exist_ok=True)

# --- 2. ACQUIRE THE LOCAL ROAD NETWORK ---
print(f"\nStep 1: Acquiring local road network within a {BUFFER_RADIUS_METERS}m radius...")
try:
    G = ox.graph_from_point((CENTER_LAT, CENTER_LON), dist=BUFFER_RADIUS_METERS, network_type='drive')
    local_streets_gdf = ox.graph_to_gdfs(G, nodes=False, edges=True)
    print(" -> Success. Local road network downloaded.")
except Exception as e:
    raise RuntimeError(f"CRITICAL FAILURE: Could not fetch local road network. Details: {e}")

# --- 3. FUZZY MATCHING LOGIC (HARDENED) ---
print(f"\nStep 2: Finding the best fuzzy match for '{TARGET_STREET_NAME}'...")
if 'name' not in local_streets_gdf.columns:
    raise KeyError("'name' column not found in the downloaded street data.")

# --- CRITICAL FIX: DATA HARDENING STEP ---
# This pre-processes the 'name' column to handle list-type entries.
# It checks each entry: if it's a list, it takes the first item; otherwise, it keeps the original string.
cleaned_names_series = local_streets_gdf['name'].apply(lambda x: x[0] if isinstance(x, list) else x)
# Now, create the "lineup" from this clean, guaranteed-string data.
available_names = cleaned_names_series.dropna().unique().tolist()
# -----------------------------------------

# Use thefuzz to find the single best match and its score
best_match, score = process.extractOne(TARGET_STREET_NAME, available_names)

if score < SCORE_CUTOFF:
    raise ValueError(
        f"CRITICAL FAILURE: No high-confidence match found for '{TARGET_STREET_NAME}'.\n"
        f"The best match was '{best_match}' with a score of only {score}%.\n"
        f"Please check your TARGET_STREET_NAME or lower the SCORE_CUTOFF."
    )

print(f" -> Best match found: '{best_match}' (Score: {score}%)")
# Use the *best match* to query the *cleaned* series for the precise query
target_street_gdf = local_streets_gdf[cleaned_names_series == best_match]
target_street_geom = target_street_gdf.geometry.unary_union

# --- 4. GENERATE JITTERED POINTS ---
print(f"\nStep 3: Generating points on the matched street geometry...")
def get_random_point_on_line(line_geometry):
    if line_geometry.geom_type == 'MultiLineString':
        line_lengths = [line.length for line in line_geometry.geoms]
        chosen_line = random.choices(line_geometry.geoms, weights=line_lengths, k=1)[0]
    else: chosen_line = line_geometry
    return chosen_line.interpolate(random.uniform(0, chosen_line.length))

random_points = [get_random_point_on_line(target_street_geom) for _ in range(NUMBER_OF_POINTS_TO_GENERATE)]
print(" -> Point generation complete.")

# --- 5. VISUALIZE FOR VERIFICATION & EXPORT ---
print("\nStep 4: Generating interactive map and exporting to HTML...")
center_point_gdf = gpd.GeoDataFrame([{'geometry': Point(CENTER_LON, CENTER_LAT)}], crs='EPSG:4326')
target_street_display_gdf = gpd.GeoDataFrame(geometry=[target_street_geom], crs='EPSG:4326')
points_gdf_display = gpd.GeoDataFrame(geometry=random_points, crs='EPSG:4326')
m = local_streets_gdf.explore(color="gray", style_kwds={"weight": 1}, name="Local Network")
target_street_display_gdf.explore(m=m, color="blue", style_kwds={"weight": 5}, name=f"Best Match: {best_match}")
points_gdf_display.explore(m=m, color="red", marker_kwds={"radius": 5}, name="Jittered Points")
center_point_gdf.explore(m=m, color="yellow", marker_kwds={"radius": 8}, name="Center Point")
try:
    m.save(HTML_OUTPUT_FILE)
    print(f" -> SUCCESS: Interactive map has been saved to: {HTML_OUTPUT_FILE}")
except Exception as e:
    print(f" -> ERROR saving HTML file: {e}")
display(m)
print("\n--- Visualization Complete ---")

Output hidden; open in https://colab.research.google.com to view.

In [3]:
# ==============================================================================
# CELL: The Definitive Jittering Engine v2.5 (Hardened against List Types)
# ==============================================================================
# Objective: A hardened engine that pre-processes OSM data to handle list-type
# names, preventing 'unhashable type' errors before fuzzy matching.

# --- 0. INSTALL/UPGRADE DEPENDENCIES ---
!pip install osmnx geopandas folium mapclassify thefuzz python-Levenshtein -q

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import osmnx as ox
import random
from thefuzz import process
from google.colab import drive
import os # <-- Import the 'os' module

print("--- Fuzzy Matching Jittering Engine (v2.5) Initializing ---")

# --- 1. CONFIGURATION ---
drive.mount('/content/drive')

# === ARCHITECT'S ACTION REQUIRED ===
CENTER_LAT = 19.369979
CENTER_LON = -99.2542798
BUFFER_RADIUS_METERS = 3000
NUMBER_OF_POINTS_TO_GENERATE = 20
TARGET_STREET_NAME = "Av Javier Barros Sierra"
SCORE_CUTOFF = 50
# ===================================

CANONICAL_PATH = '/content/drive/My Drive/_Pienza/Assets/Phase_1'
HTML_OUTPUT_FILE = os.path.join(CANONICAL_PATH, 'fuzzy_match_verification_map_v2.html')
os.makedirs(CANONICAL_PATH, exist_ok=True)

# --- 2. ACQUIRE THE LOCAL ROAD NETWORK ---
print(f"\nStep 1: Acquiring local road network within a {BUFFER_RADIUS_METERS}m radius...")
try:
    G = ox.graph_from_point((CENTER_LAT, CENTER_LON), dist=BUFFER_RADIUS_METERS, network_type='drive')
    local_streets_gdf = ox.graph_to_gdfs(G, nodes=False, edges=True)
    print(" -> Success. Local road network downloaded.")
except Exception as e:
    raise RuntimeError(f"CRITICAL FAILURE: Could not fetch local road network. Details: {e}")

# --- 3. FUZZY MATCHING LOGIC (HARDENED) ---
print(f"\nStep 2: Finding the best fuzzy match for '{TARGET_STREET_NAME}'...")
if 'name' not in local_streets_gdf.columns:
    raise KeyError("'name' column not found in the downloaded street data.")

# --- CRITICAL FIX: DATA HARDENING STEP ---
# This pre-processes the 'name' column to handle list-type entries.
# It checks each entry: if it's a list, it takes the first item; otherwise, it keeps the original string.
cleaned_names_series = local_streets_gdf['name'].apply(lambda x: x[0] if isinstance(x, list) else x)
# Now, create the "lineup" from this clean, guaranteed-string data.
available_names = cleaned_names_series.dropna().unique().tolist()
# -----------------------------------------

# Use thefuzz to find the single best match and its score
best_match, score = process.extractOne(TARGET_STREET_NAME, available_names)

if score < SCORE_CUTOFF:
    raise ValueError(
        f"CRITICAL FAILURE: No high-confidence match found for '{TARGET_STREET_NAME}'.\n"
        f"The best match was '{best_match}' with a score of only {score}%.\n"
        f"Please check your TARGET_STREET_NAME or lower the SCORE_CUTOFF."
    )

print(f" -> Best match found: '{best_match}' (Score: {score}%)")
# Use the *best match* to query the *cleaned* series for the precise query
target_street_gdf = local_streets_gdf[cleaned_names_series == best_match]
target_street_geom = target_street_gdf.geometry.unary_union

# --- 4. GENERATE JITTERED POINTS ---
print(f"\nStep 3: Generating points on the matched street geometry...")
def get_random_point_on_line(line_geometry):
    if line_geometry.geom_type == 'MultiLineString':
        line_lengths = [line.length for line in line_geometry.geoms]
        chosen_line = random.choices(line_geometry.geoms, weights=line_lengths, k=1)[0]
    else: chosen_line = line_geometry
    return chosen_line.interpolate(random.uniform(0, chosen_line.length))

random_points = [get_random_point_on_line(target_street_geom) for _ in range(NUMBER_OF_POINTS_TO_GENERATE)]
print(" -> Point generation complete.")

# --- 5. VISUALIZE FOR VERIFICATION & EXPORT ---
print("\nStep 4: Generating interactive map and exporting to HTML...")
center_point_gdf = gpd.GeoDataFrame([{'geometry': Point(CENTER_LON, CENTER_LAT)}], crs='EPSG:4326')
target_street_display_gdf = gpd.GeoDataFrame(geometry=[target_street_geom], crs='EPSG:4326')
points_gdf_display = gpd.GeoDataFrame(geometry=random_points, crs='EPSG:4326')
m = local_streets_gdf.explore(color="gray", style_kwds={"weight": 1}, name="Local Network")
target_street_display_gdf.explore(m=m, color="blue", style_kwds={"weight": 5}, name=f"Best Match: {best_match}")
points_gdf_display.explore(m=m, color="red", marker_kwds={"radius": 5}, name="Jittered Points")
center_point_gdf.explore(m=m, color="yellow", marker_kwds={"radius": 8}, name="Center Point")
try:
    m.save(HTML_OUTPUT_FILE)
    print(f" -> SUCCESS: Interactive map has been saved to: {HTML_OUTPUT_FILE}")
except Exception as e:
    print(f" -> ERROR saving HTML file: {e}")
display(m)
print("\n--- Visualization Complete ---")

Output hidden; open in https://colab.research.google.com to view.

In [2]:
# ==============================================================================
# CELL: The Forensic Visualization Engine v3.0
# ==============================================================================
# Objective: A diagnostic tool to visualize the top 5 fuzzy match candidates
# for a given search term within a local area.

# --- 0. INSTALL/UPGRADE DEPENDENCIES ---
!pip install osmnx geopandas folium mapclassify thefuzz python-Levenshtein -q

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import osmnx as ox
from thefuzz import process
from google.colab import drive
import os

print("--- Forensic Visualization Engine Initializing ---")

# --- 1. CONFIGURATION ---
drive.mount('/content/drive')

# === ARCHITECT'S ACTION REQUIRED ===
CENTER_LAT = 19.369979
CENTER_LON = -99.2542798
BUFFER_RADIUS_METERS = 3000
TARGET_STREET_NAME = "Av Javier Barros Sierra, Santa Fe"
# ===================================

CANONICAL_PATH = '/content/drive/My Drive/_Pienza/Assets/Phase_1'
HTML_OUTPUT_FILE = os.path.join(CANONICAL_PATH, 'forensic_visualization.html')
os.makedirs(CANONICAL_PATH, exist_ok=True)

# --- 2. ACQUIRE LOCAL ROAD NETWORK ---
print(f"\nStep 1: Acquiring local road network within {BUFFER_RADIUS_METERS}m...")
try:
    G = ox.graph_from_point((CENTER_LAT, CENTER_LON), dist=BUFFER_RADIUS_METERS, network_type='drive')
    local_streets_gdf = ox.graph_to_gdfs(G, nodes=False, edges=True)
    local_streets_gdf['clean_name'] = local_streets_gdf['name'].apply(lambda x: x[0] if isinstance(x, list) else x)
    available_names = local_streets_gdf['clean_name'].dropna().unique().tolist()
    print(" -> Success. Local road network downloaded.")
except Exception as e:
    raise RuntimeError(f"CRITICAL FAILURE: Could not fetch local road network. Details: {e}")

# --- 3. FUZZY MATCHING FORENSICS ---
print(f"\nStep 2: Performing forensic fuzzy match for '{TARGET_STREET_NAME}'...")
if not available_names:
    raise ValueError("No named streets were found in the specified buffer zone.")

# Get the Top 5 best matches
top_5_matches = process.extract(TARGET_STREET_NAME, available_names, limit=5)

print("\n--- FORENSIC MATCH REPORT ---")
match_geometries = []
for i, (match_name, score) in enumerate(top_5_matches):
    print(f"  Candidate #{i+1}: '{match_name}' (Score: {score}%)")
    match_geom = local_streets_gdf[local_streets_gdf['clean_name'] == match_name].geometry.unary_union
    match_geometries.append({
        'rank': f"#{i+1}: {match_name} ({score}%)",
        'geometry': match_geom,
        'is_best_match': 'Best Match' if i == 0 else 'Other Candidate'
    })
print("---------------------------\n")

# --- 4. VISUALIZE ALL EVIDENCE ---
print("Step 3: Generating interactive map with all candidates...")
candidates_gdf = gpd.GeoDataFrame(match_geometries, crs="EPSG:4326")
center_point_gdf = gpd.GeoDataFrame([{'geometry': Point(CENTER_LON, CENTER_LAT)}], crs='EPSG:4326')

# Create the base map of the full local network
m = local_streets_gdf.explore(color="lightgray", style_kwds={"weight": 2}, name="Local Network")

# Overlay the Top 5 candidates, styled differently for the best match
candidates_gdf.explore(
    m=m,
    column="is_best_match",
    categorical=True,
    cmap={"Best Match": "red", "Other Candidate": "blue"},
    tooltip="rank", # Show the rank, name, and score on hover
    popup=True,
    style_kwds={"weight": 5, "opacity": 0.8}
)
center_point_gdf.explore(m=m, color="yellow", marker_kwds={"radius": 8}, name="Center Point")

try:
    m.save(HTML_OUTPUT_FILE)
    print(f" -> SUCCESS: Forensic map has been saved to: {HTML_OUTPUT_FILE}")
except Exception as e:
    print(f" -> ERROR saving HTML file: {e}")

display(m)
print("\n--- FORENSIC VISUALIZATION COMPLETE ---")
print("Map shows: Center (Yellow), Local Network (Gray), Best Match (Red), Other Candidates (Blue)")

--- Forensic Visualization Engine Initializing ---
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Step 1: Acquiring local road network within 3000m...
 -> Success. Local road network downloaded.

Step 2: Performing forensic fuzzy match for 'Av Javier Barros Sierra, Santa Fe'...

--- FORENSIC MATCH REPORT ---
  Candidate #1: 'Puerta Santa Fe' (Score: 86%)
  Candidate #2: 'Avenida Santa Fe' (Score: 86%)
  Candidate #3: 'Calle Veredas Santa Fe' (Score: 86%)
  Candidate #4: 'Vereda Santa Fe' (Score: 86%)
  Candidate #5: 'Camino a Santa Fe' (Score: 86%)
---------------------------

Step 3: Generating interactive map with all candidates...


/tmp/ipython-input-2397224997.py:57: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  match_geom = local_streets_gdf[local_streets_gdf['clean_name'] == match_name].geometry.unary_union
/tmp/ipython-input-2397224997.py:57: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  match_geom = local_streets_gdf[local_streets_gdf['clean_name'] == match_name].geometry.unary_union
/tmp/ipython-input-2397224997.py:57: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  match_geom = local_streets_gdf[local_streets_gdf['clean_name'] == match_name].geometry.unary_union
/tmp/ipython-input-2397224997.py:57: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  match_geom = local_streets_gdf[local_streets_gdf['clean_name'] == match_name].geometry.unary_union
/tmp/ipython-input-2397224997.py:57: Depreca

IndexError: index 1 is out of bounds for axis 0 with size 1